In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="cXVV91ELxDgKnbtwz6yM")
project = rf.workspace("cristhopers-workspace").project("pavescan-rqsfz")
version = project.version(4)
dataset = version.download("png-mask-semantic")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PAVESCAN-4 in png-mask-semantic:: 100%|██████████| 3701/3701 [00:00<00:00, 8512.40it/s]


In [ ]:
print(dataset.location)

/content/PAVESCAN-4


In [ ]:
!pip install -q --upgrade "transformers==4.44.2" "accelerate>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import transformers
print(transformers.__version__)


4.44.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUTA_GUARDADO_DRIVE = '/content/drive/MyDrive/PAVESCAN/modelo_segformer'
os.makedirs(RUTA_GUARDADO_DRIVE, exist_ok=True)


Mounted at /content/drive


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset

class PavescanDataset(Dataset):
    def __init__(self, carpeta, processor):
        self.carpeta = carpeta
        self.processor = processor
        self.pares = []
        for nombre in sorted(os.listdir(carpeta)):
            if nombre.endswith('.jpg'):
                base = nombre[:-4]
                ruta_mask = os.path.join(carpeta, base + '_mask.png')
                if os.path.isfile(ruta_mask):
                    self.pares.append((
                        os.path.join(carpeta, nombre),
                        ruta_mask
                    ))
        print(f'  {os.path.basename(carpeta)}: {len(self.pares)} pares imagen/máscara')

    def __len__(self):
        return len(self.pares)

    def __getitem__(self, idx):
        ruta_img, ruta_mask = self.pares[idx]
        imagen = Image.open(ruta_img).convert('RGB')
        mascara = Image.open(ruta_mask)
        mascara = np.array(mascara)
        mascara = Image.fromarray(mascara.astype(np.uint8))

        entradas = self.processor(imagen, mascara, return_tensors='pt')
        for k in entradas:
            entradas[k] = entradas[k].squeeze(0)
        return entradas


In [ ]:
from transformers import SegformerImageProcessor

CHECKPOINT = 'nvidia/segformer-b2-finetuned-ade-512-512'

processor = SegformerImageProcessor.from_pretrained(CHECKPOINT)
processor.do_reduce_labels = False
processor.size = {'height': 512, 'width': 512}

train_dataset = PavescanDataset(os.path.join(dataset.location, 'train'), processor)
valid_dataset = PavescanDataset(os.path.join(dataset.location, 'valid'), processor)
test_dataset  = PavescanDataset(os.path.join(dataset.location, 'test'),  processor)



The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

  train: 1594 pares imagen/máscara
  valid: 169 pares imagen/máscara
  test: 85 pares imagen/máscara


The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type'


In [ ]:
from transformers import SegformerForSemanticSegmentation

id2label = {0: 'background', 1: 'bache'}
label2id = {v: k for k, v in id2label.items()}

model = SegformerForSemanticSegmentation.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/110M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b2-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 768, 1, 1]) in the checkpoint and torch.Size([2, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import evaluate
import torch.nn.functional as F

metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits_tensor = torch.from_numpy(logits)
    logits_tensor = F.interpolate(
        logits_tensor, size=labels.shape[-2:],
        mode='bilinear', align_corners=False
    ).argmax(dim=1)

    pred_labels = logits_tensor.numpy()
    resultado = metric.compute(
        predictions=pred_labels,
        references=labels,
        num_labels=2,
        ignore_index=255,
        reduce_labels=False
    )
    return {
        'mean_iou': resultado['mean_iou'],
        'mean_accuracy': resultado['mean_accuracy'],
    }


ModuleNotFoundError: No module named 'evaluate'

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


In [ ]:
import evaluate
import torch.nn.functional as F

metric = evaluate.load('mean_iou')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits_tensor = torch.from_numpy(logits)
    logits_tensor = F.interpolate(
        logits_tensor, size=labels.shape[-2:],
        mode='bilinear', align_corners=False
    ).argmax(dim=1)

    pred_labels = logits_tensor.numpy()
    resultado = metric.compute(
        predictions=pred_labels,
        references=labels,
        num_labels=2,
        ignore_index=255,
        reduce_labels=False
    )
    return {
        'mean_iou': resultado['mean_iou'],
        'mean_accuracy': resultado['mean_accuracy'],
    }


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class DiceCELoss(nn.Module):
    """
    Combina Cross-Entropy ponderada (más peso a la clase 'bache', minoritaria)
    con Dice Loss (optimiza directamente la superposición, similar al IoU).
    Objetivo: mejorar el Recall sin destruir la Precisión ya alcanzada.
    """
    def __init__(self, weight_dice=0.6, weight_ce=0.4,
                 peso_background=1.0, peso_bache=3.0):
        super().__init__()
        self.weight_dice = weight_dice
        self.weight_ce = weight_ce
        # Más peso a 'bache' (clase 1) para compensar el desbalance
        self.class_weights = torch.tensor([peso_background, peso_bache])

    def forward(self, logits, labels):
        weights = self.class_weights.to(logits.device).type_as(logits)
        ce_loss = F.cross_entropy(logits, labels, weight=weights, ignore_index=255)

        probs = F.softmax(logits, dim=1)[:, 1, :, :]  # canal "bache"
        labels_bin = (labels == 1).float()
        intersection = (probs * labels_bin).sum()
        dice = (2. * intersection + 1e-6) / (probs.sum() + labels_bin.sum() + 1e-6)
        dice_loss = 1 - dice

        return self.weight_ce * ce_loss + self.weight_dice * dice_loss


class PavescanTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        # Los logits salen en resolución reducida (H/4, W/4) — reescalar
        # al tamaño real de las máscaras (512x512) antes de comparar.
        logits = F.interpolate(outputs.logits, size=labels.shape[-2:],
                               mode='bilinear', align_corners=False)

        loss_fn = DiceCELoss()
        loss = loss_fn(logits, labels)

        if return_outputs:
            outputs.loss = loss
        return (loss, outputs) if return_outputs else loss


In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

args = TrainingArguments(
    output_dir='/content/pavescan_checkpoints',
    learning_rate=6e-5,              # estándar validado para fine-tuning SegFormer
    num_train_epochs=100,            # techo alto — el early stopping cortará antes
    per_device_train_batch_size=4,   # seguro para T4 gratuita
    per_device_eval_batch_size=4,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model='mean_iou',
    greater_is_better=True,
    report_to='none',
)

trainer = PavescanTrainer(           # ← Único cambio importante
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

trainer.train()


Epoch,Training Loss,Validation Loss,Mean Iou,Mean Accuracy
1,0.488000,0.556900,0.660916,0.717118
2,0.493600,0.462090,0.754247,0.911705
3,0.377600,0.490656,0.740399,0.802369
4,0.311900,0.540571,0.711320,0.743441
5,0.435600,0.514680,0.763735,0.806454
6,0.294300,0.591027,0.694498,0.723299
7,0.394800,0.443317,0.832625,0.900751
8,0.253600,0.519564,0.763691,0.819322
9,0.376400,0.656023,0.682535,0.706923
10,0.272800,0.510419,0.777698,0.818270


Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
Downcasting array dtype int64 to int32 to be compatible with 'Pillow'


TrainOutput(global_step=4788, training_loss=0.39620047683205917, metrics={'train_runtime': 4306.6654, 'train_samples_per_second': 37.012, 'train_steps_per_second': 9.265, 'total_flos': 2.4683692868889477e+18, 'train_loss': 0.39620047683205917, 'epoch': 12.0})

In [ ]:
resultados_test = trainer.evaluate(test_dataset)
print(resultados_test)


Downcasting array dtype int64 to int32 to be compatible with 'Pillow'


{'eval_loss': 0.529131293296814, 'eval_mean_iou': 0.7320200850176304, 'eval_mean_accuracy': 0.8400960424001842, 'eval_runtime': 15.8041, 'eval_samples_per_second': 5.378, 'eval_steps_per_second': 1.392, 'epoch': 12.0}


In [ ]:
RUTA_GUARDADO_DRIVE_DICE = '/content/drive/MyDrive/PAVESCAN/modelo_segformer_dice'
import os
os.makedirs(RUTA_GUARDADO_DRIVE_DICE, exist_ok=True)

model.save_pretrained('/content/modelo_segformer_dice')
processor.save_pretrained('/content/modelo_segformer_dice')

!cp -r /content/modelo_segformer_dice/* "{RUTA_GUARDADO_DRIVE_DICE}/"
print('Modelo guardado en Drive:', RUTA_GUARDADO_DRIVE_DICE)


Modelo guardado en Drive: /content/drive/MyDrive/PAVESCAN/modelo_segformer_dice
